# Estimate Delta 01 — baseline regression

Full-sample robustness specification for June 1 through November 4, 2024:

\[
r_{TAN,t} = \alpha + \beta\,\Delta p_t + \gamma\,r_{SPY,t} + \varepsilon_t
\]

Inference uses Newey–West (HAC) standard errors with a Bartlett kernel, five lags, and finite-sample correction. This is a robustness estimate rather than the headline result because measurement noise in daily \(\Delta p\) can attenuate \(\beta\).

In [ ]:
from pathlib import Path

import pandas as pd
import statsmodels.api as sm

DATA_PATH = Path("data/daily_merged_tan_spy_polymarket_2024.csv")
RESULTS_PATH = Path("data/01_estimate_delta_baseline_results.csv")
START_DATE = pd.Timestamp("2024-06-01")
END_DATE = pd.Timestamp("2024-11-04")

In [ ]:
daily = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

regression_data = daily.loc[
    START_DATE:END_DATE,
    ["TAN ret", "SPY ret", "Δp"],
].dropna()

assert not regression_data.empty
assert regression_data.index.min() >= START_DATE
assert regression_data.index.max() <= END_DATE
assert regression_data.index.is_unique

print(f"Observations: {len(regression_data)}")
print(f"Effective coverage: {regression_data.index.min().date()} to {regression_data.index.max().date()}")
regression_data.describe()

In [ ]:
y = regression_data["TAN ret"]
X = sm.add_constant(regression_data[["Δp", "SPY ret"]], has_constant="add")

baseline_model = sm.OLS(y, X).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 5,
        "kernel": "bartlett",
        "use_correction": True,
    },
    use_t=True,
)

baseline_model.summary()

In [ ]:
baseline_result = pd.DataFrame(
    {
        "specification": ["Baseline robustness: TAN on Δp and SPY"],
        "sample_start": [START_DATE.date().isoformat()],
        "sample_end": [END_DATE.date().isoformat()],
        "n_obs": [int(baseline_model.nobs)],
        "newey_west_lags": [5],
        "beta_delta_p": [baseline_model.params["Δp"]],
        "se_beta_delta_p": [baseline_model.bse["Δp"]],
        "p_value_beta_delta_p": [baseline_model.pvalues["Δp"]],
        "alpha": [baseline_model.params["const"]],
        "gamma_spy": [baseline_model.params["SPY ret"]],
        "r_squared": [baseline_model.rsquared],
    }
)

baseline_result.to_csv(RESULTS_PATH, index=False)

beta = baseline_result.loc[0, "beta_delta_p"]
beta_se = baseline_result.loc[0, "se_beta_delta_p"]
print(f"β = {beta:.6f}")
print(f"Newey–West SE(5) = {beta_se:.6f}")
print(f"Saved results to {RESULTS_PATH.resolve()}")

baseline_result

## Recorded robustness estimate

- **β on Δp:** −0.331311
- **Newey–West SE (5 lags):** 0.100980
- Observations: 108

This full-sample estimate is retained as the robustness column; noise in daily Δp can attenuate the coefficient.

## High-signal event-window regression

For each configured pre-election shock, the prediction-market change and ETF returns are measured over the same window. All three shocks occurred outside regular U.S. trading hours, so each window runs from the prior NYSE close to the next NYSE open. ETF gaps are adjusted for corporate-action factors. The event observations are pooled with the top decile of absolute daily `Δp` in the June 1–November 4 sample; reaction-date daily observations are excluded to avoid counting the same shock twice.

Event timestamps are sourced from **Tsang & Yang, arXiv:2603.03152**, as recorded in `config/events.json`.

In [ ]:
import json
from datetime import time as clock_time
from zoneinfo import ZoneInfo

CONFIG_PATH = Path("config/events.json")
ETF_PATH = Path("data/tan_spy_daily_2016_2024.csv")
POLYMARKET_PATH = Path("data/polymarket_trump_2024_yes_1min.csv")
HEADLINE_RESULTS_PATH = Path("data/01_estimate_delta_high_signal_results.csv")
EVENT_WINDOWS_PATH = Path("data/01_estimate_delta_event_windows.csv")

with CONFIG_PATH.open() as config_file:
    event_config = json.load(config_file)

etf_prices = pd.read_csv(ETF_PATH, parse_dates=["Date"])
polymarket_1min = pd.read_csv(POLYMARKET_PATH, parse_dates=["timestamp_utc"]).sort_values("timestamp_utc")

NY = ZoneInfo("America/New_York")
PRE_ELECTION_EVENTS = [
    "presidential_debate",
    "trump_assassination_attempt",
    "biden_dropout",
]

print("Timestamp source:", event_config["timestamp_source"])
print("Events:", PRE_ELECTION_EVENTS)

In [ ]:
trading_dates = pd.DatetimeIndex(
    sorted(
        set(etf_prices.loc[etf_prices["Ticker"] == "TAN", "Date"])
        & set(etf_prices.loc[etf_prices["Ticker"] == "SPY", "Date"])
    )
)

price_tables = {}
for ticker in ["TAN", "SPY"]:
    table = etf_prices.loc[etf_prices["Ticker"] == ticker].set_index("Date").sort_index().copy()
    table["Adj Open"] = table["Open"] * table["Adj Close"] / table["Close"]
    price_tables[ticker] = table

polymarket_price = polymarket_1min.set_index("timestamp_utc")["price"]


def ny_timestamp(date, hour, minute):
    return pd.Timestamp(
        year=date.year,
        month=date.month,
        day=date.day,
        hour=hour,
        minute=minute,
        tz=NY,
    ).tz_convert("UTC")


def build_outside_hours_event(event_key):
    event = event_config["events"][event_key]
    event_ts = pd.Timestamp(event["timestamp_utc"]).tz_convert("UTC")
    event_local = event_ts.tz_convert(NY)
    event_date = pd.Timestamp(event_local.date())
    is_trading_date = event_date in trading_dates
    hit_during_us_hours = (
        is_trading_date
        and clock_time(9, 30) <= event_local.time() < clock_time(16, 0)
    )
    if hit_during_us_hours:
        raise ValueError(f"{event_key} requires intraday ETF data for its first-two-hours window.")

    eligible_dates = trading_dates[trading_dates > event_date]
    if is_trading_date and event_local.time() < clock_time(9, 30):
        reaction_date = event_date
    else:
        reaction_date = eligible_dates[0]

    reaction_position = trading_dates.get_loc(reaction_date)
    prior_date = trading_dates[reaction_position - 1]
    window_start = ny_timestamp(prior_date.date(), 16, 0)
    window_end = ny_timestamp(reaction_date.date(), 9, 30)

    p_start = polymarket_price.asof(window_start)
    p_end = polymarket_price.asof(window_end)
    ticker_returns = {
        f"{ticker} ret": (
            price_tables[ticker].loc[reaction_date, "Adj Open"]
            / price_tables[ticker].loc[prior_date, "Adj Close"]
            - 1
        )
        for ticker in ["TAN", "SPY"]
    }

    return {
        "event": event_key,
        "label": event["label"],
        "event_timestamp_utc": event_ts,
        "Date": reaction_date,
        "window_start_utc": window_start,
        "window_end_utc": window_end,
        **ticker_returns,
        "TAN market-adjusted ret": ticker_returns["TAN ret"] - ticker_returns["SPY ret"],
        "p_start": p_start,
        "p_end": p_end,
        "Δp": p_end - p_start,
        "observation_type": "event_window",
    }


event_windows = pd.DataFrame(build_outside_hours_event(key) for key in PRE_ELECTION_EVENTS)
event_windows.to_csv(EVENT_WINDOWS_PATH, index=False)
event_windows

In [ ]:
signal_cutoff = regression_data["Δp"].abs().quantile(0.90)
reaction_dates = set(event_windows["Date"])

top_decile_days = (
    regression_data.loc[regression_data["Δp"].abs() >= signal_cutoff]
    .loc[lambda frame: ~frame.index.isin(reaction_dates)]
    .reset_index()
)
top_decile_days["observation_type"] = "top_decile_day"
top_decile_days["event"] = pd.NA

event_regression_rows = event_windows[
    ["Date", "TAN ret", "SPY ret", "Δp", "observation_type", "event"]
]

high_signal = (
    pd.concat(
        [
            top_decile_days[
                ["Date", "TAN ret", "SPY ret", "Δp", "observation_type", "event"]
            ],
            event_regression_rows,
        ],
        ignore_index=True,
    )
    .sort_values(["Date", "observation_type"])
    .reset_index(drop=True)
)

print(f"Full-sample |Δp| 90th-percentile cutoff: {signal_cutoff:.6f}")
print(f"Top-decile daily observations retained: {len(top_decile_days)}")
print(f"Event-window observations: {len(event_regression_rows)}")
high_signal

In [ ]:
headline_y = high_signal["TAN ret"]
headline_X = sm.add_constant(high_signal[["Δp", "SPY ret"]], has_constant="add")

headline_model = sm.OLS(headline_y, headline_X).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 5,
        "kernel": "bartlett",
        "use_correction": True,
    },
    use_t=True,
)

headline_model.summary()

In [ ]:
headline_result = pd.DataFrame(
    {
        "specification": ["High-signal headline: event windows plus top-decile |Δp| days"],
        "sample_start": [START_DATE.date().isoformat()],
        "sample_end": [END_DATE.date().isoformat()],
        "n_obs": [int(headline_model.nobs)],
        "event_window_obs": [len(event_regression_rows)],
        "top_decile_day_obs": [len(top_decile_days)],
        "abs_delta_p_cutoff": [signal_cutoff],
        "newey_west_lags": [5],
        "beta_delta_p": [headline_model.params["Δp"]],
        "se_beta_delta_p": [headline_model.bse["Δp"]],
        "p_value_beta_delta_p": [headline_model.pvalues["Δp"]],
        "alpha": [headline_model.params["const"]],
        "gamma_spy": [headline_model.params["SPY ret"]],
        "r_squared": [headline_model.rsquared],
        "delta_E_for_0_to_1": [headline_model.params["Δp"]],
    }
)
headline_result.to_csv(HEADLINE_RESULTS_PATH, index=False)

headline_beta = headline_result.loc[0, "beta_delta_p"]
headline_se = headline_result.loc[0, "se_beta_delta_p"]
print(f"Headline β = ΔE for a 0→1 probability jump = {headline_beta:.6f}")
print(f"Newey–West SE(5) = {headline_se:.6f}")
print(f"Saved event windows to {EVENT_WINDOWS_PATH.resolve()}")
print(f"Saved headline result to {HEADLINE_RESULTS_PATH.resolve()}")

headline_result

## Recorded headline estimate

- **Headline β = ΔE for a 0→1 probability jump:** −0.234202
- **Newey–West SE (5 lags):** 0.112716
- High-signal observations: 15 (3 event windows + 12 top-decile days, including ties at the 90th-percentile cutoff)
- Two-sided p-value: 0.059855

The baseline full-sample estimate remains the robustness result; this high-signal estimate is the headline result.

## Comparable-event anchor: 2016 election

Compute TAN's adjusted overnight gap from the November 8, 2016 close to the November 9 open, subtract SPY's corresponding gap, and scale by the supplied approximate probability surprise of **0.75**:

\[
\Delta E_{2016} = \frac{r^{gap}_{TAN} - r^{gap}_{SPY}}{0.75}.
\]

The 0.75 surprise is retained as the stated calibration assumption. For clarity, a literal move from 0.15 to 1.00 would equal 0.85, so the calculation below does not infer the surprise from those rounded endpoints.

In [ ]:
ANCHOR_RESULTS_PATH = Path("data/01_estimate_delta_2016_anchor.csv")
ANCHOR_PRIOR_DATE = pd.Timestamp("2016-11-08")
ANCHOR_REACTION_DATE = pd.Timestamp("2016-11-09")
PROBABILITY_SURPRISE_2016 = 0.75

anchor_gaps = {}
for ticker in ["TAN", "SPY"]:
    anchor_gaps[ticker] = (
        price_tables[ticker].loc[ANCHOR_REACTION_DATE, "Adj Open"]
        / price_tables[ticker].loc[ANCHOR_PRIOR_DATE, "Adj Close"]
        - 1
    )

market_adjusted_gap_2016 = anchor_gaps["TAN"] - anchor_gaps["SPY"]
delta_E_2016 = market_adjusted_gap_2016 / PROBABILITY_SURPRISE_2016

anchor_result = pd.DataFrame(
    {
        "prior_close_date": [ANCHOR_PRIOR_DATE.date().isoformat()],
        "reaction_open_date": [ANCHOR_REACTION_DATE.date().isoformat()],
        "tan_adjusted_gap": [anchor_gaps["TAN"]],
        "spy_adjusted_gap": [anchor_gaps["SPY"]],
        "tan_market_adjusted_gap": [market_adjusted_gap_2016],
        "probability_surprise": [PROBABILITY_SURPRISE_2016],
        "delta_E_2016": [delta_E_2016],
    }
)
anchor_result.to_csv(ANCHOR_RESULTS_PATH, index=False)

print(f"TAN adjusted gap: {anchor_gaps['TAN']:.6f}")
print(f"SPY adjusted gap: {anchor_gaps['SPY']:.6f}")
print(f"TAN market-adjusted gap: {market_adjusted_gap_2016:.6f}")
print(f"ΔE_2016: {delta_E_2016:.6f}")
print(f"Saved anchor result to {ANCHOR_RESULTS_PATH.resolve()}")

anchor_result

## Recorded 2016 anchor

- TAN adjusted overnight gap: **−0.092789**
- SPY adjusted overnight gap: **−0.008127**
- TAN market-adjusted gap: **−0.084662**
- Assumed probability surprise: **0.75**
- **ΔE_2016: −0.112883**

## Final comparison

The table below assembles all three estimates. The 2016 anchor has no independently estimable standard error: it is one event divided by an assumed probability surprise, so reporting a regression-style SE would imply unsupported precision. It is retained as a calibration anchor with `NaN` in the SE column.

In [ ]:
FINAL_TABLE_PATH = Path("data/01_estimate_delta_final_comparison.csv")

final_comparison = pd.DataFrame(
    [
        {
            "estimate": "Full-sample OLS",
            "delta_E": baseline_result.loc[0, "beta_delta_p"],
            "standard_error": baseline_result.loc[0, "se_beta_delta_p"],
            "n_obs": int(baseline_result.loc[0, "n_obs"]),
            "inference": "Newey–West, 5 lags",
            "role": "Robustness",
        },
        {
            "estimate": "Event-window high-signal subsample",
            "delta_E": headline_result.loc[0, "beta_delta_p"],
            "standard_error": headline_result.loc[0, "se_beta_delta_p"],
            "n_obs": int(headline_result.loc[0, "n_obs"]),
            "inference": "Newey–West, 5 lags",
            "role": "Headline / sizing",
        },
        {
            "estimate": "2016 comparable-event anchor",
            "delta_E": anchor_result.loc[0, "delta_E_2016"],
            "standard_error": float("nan"),
            "n_obs": 1,
            "inference": "SE not identifiable from one event",
            "role": "Calibration anchor",
        },
    ]
)

estimate_low = final_comparison["delta_E"].min()
estimate_high = final_comparison["delta_E"].max()
estimate_range_width = estimate_high - estimate_low
sizing_delta_E = headline_result.loc[0, "beta_delta_p"]

final_comparison.to_csv(FINAL_TABLE_PATH, index=False)

print(f"ΔE range: [{estimate_low:.6f}, {estimate_high:.6f}]")
print(f"Range width: {estimate_range_width:.6f}")
print(f"Sizing estimate (event-window): {sizing_delta_E:.6f}")
print(f"Saved final table to {FINAL_TABLE_PATH.resolve()}")

final_comparison

## Final readout

All three estimates are negative, so they agree on direction, but their magnitudes are materially dispersed:

- Full-sample OLS: **−0.331311** (SE **0.100980**)
- Event-window high-signal subsample: **−0.234202** (SE **0.112716**)
- 2016 comparable-event anchor: **−0.112883** (SE not identifiable)

The cross-method range is **[−0.331311, −0.112883]**, with width **0.218428**. This dispersion is itself a finding about design and signal quality. Consistent with the pre-specified hierarchy, use the event-window estimate **ΔE = −0.234202** for sizing.